# Objetivo de Notebook

Esta notebook implementa un pipeline robusto para modelado no supervisado, separando claramente los datos de entrenamiento de los datos de evaluación, asegurando que el modelo no vea los mismos pozos/etapas que luego va a analizar.

## 🤖 Split Robusto y Modelado No Supervisado - CCL Anomalías

Esta notebook implementa:
- División del dataset por pozo y etapa.
- Entrenamiento del modelo (Isolation Forest) en runs seleccionadas.
- Evaluación del modelo en datos no vistos (evaluación visual de anomalías).
- Preparación de los resultados para visualización.

Esta estrategia permite evitar sobreajuste y simular condiciones de operación real.


### 💾 Celda 2 – Carga del dataset enriquecido

In [109]:
import pandas as pd

# Cargar dataset con features + CCL + TENS
df = pd.read_csv(r"C:\Developer\fundamentos\data\ccl_features.csv")
df = df.sort_values(by=["pozo", "etapa", "DEPT"]).reset_index(drop=True)

print(f"Total registros: {len(df)}")
df.head()


Total registros: 502890


,DEPT,CCL,TENS,archivo_origen,pozo,sentido,etapa,CCL_norm,dCCL,abs_dCCL,...,CCL_norm_mean,CCL_norm_std,CCL_norm_max,CCL_norm_min,abs_dCCL_mean,abs_dCCL_std,abs_dCCL_max,TENS_mean,TENS_std,TENS_max
0,2800.0452,-0.00787,740.99995,BPO-2702_E1_Down__25Jan25_002239.las,BPO-2702,Down,E1,-1.575891,NaN,NaN,...,0.142625,1.904589,10.0,-9.993993,1.990218,1.726621,16.770124,784.461514,82.311949,1290.00003
1,2800.1976,-0.00774,742.00013,BPO-2702_E1_Down__25Jan25_002239.las,BPO-2702,Down,E1,-1.549860,0.026031,0.026031,...,0.142625,1.904589,10.0,-9.993993,1.990218,1.726621,16.770124,784.461514,82.311949,1290.00003
2,2800.3500,0.01254,744.99998,BPO-2702_E1_Down__25Jan25_002239.las,BPO-2702,Down,E1,2.511013,4.060873,4.060873,...,0.142625,1.904589,10.0,-9.993993,1.990218,1.726621,16.770124,784.461514,82.311949,1290.00003
3,2800.5024,0.01011,744.00003,BPO-2702_E1_Down__25Jan25_002239.las,BPO-2702,Down,E1,2.024429,-0.486584,0.486584,...,0.142625,1.904589,10.0,-9.993993,1.990218,1.726621,16.770124,784.461514,82.311949,1290.00003
4,2800.6548,0.00512,744.00003,BPO-2702_E1_Down__25Jan25_002239.las,BPO-2702,Down,E1,1.025230,-0.999199,0.999199,...,0.142625,1.904589,10.0,-9.993993,1.990218,1.726621,16.770124,784.461514,82.311949,1290.00003


### 📋 Celda 3 – Generar resumen de etapas por pozo

In [110]:
df_summary = df[["pozo", "etapa"]].drop_duplicates().sort_values(["pozo", "etapa"])
print(f"Total de runs únicas: {len(df_summary)}")
df_summary.head(49)


Total de runs únicas: 43


,pozo,etapa
0,BPO-2702,E1
20312,BPO-2702,E10
37222,BPO-2702,E11
53855,BPO-2702,E12
70151,BPO-2702,E17
84331,BPO-2702,E18
97647,BPO-2702,E19
111407,BPO-2702,E2
131339,BPO-2702,E20
144712,BPO-2702,E21


### ✂️ Celda 4 – Split robusto por combinación pozo-etapa

In [137]:
# ⚙️ Definí manualmente qué pozo(s) y etapa(s) usar
train_seleccion = [
    ("BPO-2702", "E12"),
    ("BPO-2702", "E18"),
]

eval_seleccion = [
    ("BPO-2702", "E19"),
]

# Separar el dataset según las combinaciones seleccionadas
df_train = df[df[["pozo", "etapa"]].apply(tuple, axis=1).isin(train_seleccion)].copy()
df_eval = df[df[["pozo", "etapa"]].apply(tuple, axis=1).isin(eval_seleccion)].copy()

print(f"✅ Train: {df_train.shape[0]} registros en {len(train_seleccion)} runs")
print(f"✅ Eval:  {df_eval.shape[0]} registros en {len(eval_seleccion)} runs")

✅ Train: 29612 registros en 2 runs
✅ Eval:  13760 registros en 1 runs


### 🧠 Celda 5 – Entrenamiento con Isolation Forest

In [138]:
from sklearn.ensemble import IsolationForest

# Seleccionar features numéricas (excluimos DEPT y cualquier score anterior)
exclude = ["DEPT", "score_iso", "anomaly_iso"]
features_cols = [col for col in df_train.select_dtypes(include='number').columns if col not in exclude]

X_train = df_train[features_cols].fillna(0)
X_eval = df_eval[features_cols].fillna(0)

iso = IsolationForest(n_estimators=100, contamination=0.02, random_state=42)
iso.fit(X_train)

# Aplicar a evaluación
df_eval["score_iso"] = -iso.decision_function(X_eval)
df_eval["anomaly_iso"] = iso.predict(X_eval)  # -1 = anomalía, 1 = normal


### 📦 Celda 6 – Guardar resultados de evaluación

In [139]:
df_eval.to_csv(r"C:\Developer\fundamentos\data\ccl_eval_scores.csv", index=False)
print("✅ Resultados de evaluación exportados a ccl_eval_scores.csv")


✅ Resultados de evaluación exportados a ccl_eval_scores.csv


### 📊 Celda 7 – Vista rápida de anomalías detectadas

In [140]:
anomalias_por_etapa = df_eval[df_eval["anomaly_iso"] == -1].groupby(["pozo", "etapa"]).size().reset_index(name="anomalías")
anomalias_por_etapa.sort_values("anomalías", ascending=False).head(10)


,pozo,etapa,anomalías
0,BPO-2702,E19,240
